# 05 Operational Weather Feature Selector

This notebook creates the ML-ready operational weather feature view from notebook 04's canonical forecast output. It does not build a separate synthetic forecast or ensemble simulation. Its role is to select the main operational weather window, validate feature availability, and write compact audit tables for notebook 06.


## Pipeline position

This notebook follows `notebooks/04_synthetic_weather_model.ipynb` and precedes `notebooks/06_build_ml_panel.ipynb`. Notebook 04 is the canonical operational forecast-weather source. Forecast-error diagnostics remain separate and are not merged into the ML-ready weather feature view.


## Inputs

The required canonical input is `data/processed/weather_forecast_operational_windows.parquet`. `data/processed/weather_forecast_error_diagnostics_windows.parquet` is diagnostics-only and used for schema/audit awareness. Deprecated legacy inputs from the older notebook 05 path are no longer canonical dependencies: `data/interim/weather_forecast_scaffold.parquet`, `data/interim/forecast_error_table.parquet`, `data/interim/forecast_sigma_tables_baseline.parquet`, and `data/processed/weather_forecast_simulated_baseline.parquet`.


## Outputs

The main output is `data/processed/weather_forecast_operational_ml_features.parquet`. The notebook also writes audit outputs under `outputs/weather_ensemble_features/`:

- `outputs/weather_ensemble_features/05_input_schema_audit.csv`
- `outputs/weather_ensemble_features/05_ml_weather_feature_groups.csv`
- `outputs/weather_ensemble_features/05_ml_weather_feature_groups.json`
- `outputs/weather_ensemble_features/05_operational_window_summary.csv`
- `outputs/weather_ensemble_features/05_horizon_source_summary.csv`
- `outputs/weather_ensemble_features/05_missingness_check.csv`
- `outputs/weather_ensemble_features/05_leakage_column_check.csv`
- `outputs/weather_ensemble_features/05_validation_checks.csv`
- `outputs/weather_ensemble_features/05_end_sample_row_reduction_audit.csv`


## Feature contract

The main operational window is `trade_08_19`. M2 uses point forecast weather only: `temp_fcst_mean`, `precip_fcst_sum`, `wind_fcst_mean`, `humid_fcst_mean`, and `cloud_fcst_mean`. M4 uses the same point forecast features plus calibrated uncertainty, probability, and interval features from notebook 04. `apparent_temp_fcst_mean` is deliberately excluded from M2 and M4. Forecast-source and fallback columns are retained for audit or stratification, not as default ML features.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

MARKER_FILES = ("README_FOR_CODEX.md", "AGENTS.md", "pyproject.toml")


def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if any((candidate / marker).exists() for marker in MARKER_FILES):
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Start Jupyter from inside the project folder "
        "or make sure README_FOR_CODEX.md, AGENTS.md, or pyproject.toml exists."
    )


def project_relative(path: Path) -> str:
    try:
        return str(path.resolve().relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


def require_file(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f"Missing required {label}: {project_relative(path)}")


def require_columns(frame: pd.DataFrame, columns: list[str], label: str):
    missing = [col for col in columns if col not in frame.columns]
    if missing:
        available = ", ".join(frame.columns.astype(str).tolist())
        raise KeyError(
            f"{label} is missing required columns {missing}. Available columns: {available}"
        )


PROJECT_ROOT = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "weather_ensemble_features"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OPERATIONAL_FORECAST_PATH = PROCESSED_DIR / "weather_forecast_operational_windows.parquet"
ERROR_DIAGNOSTICS_PATH = PROCESSED_DIR / "weather_forecast_error_diagnostics_windows.parquet"
ML_WEATHER_FEATURES_PATH = PROCESSED_DIR / "weather_forecast_operational_ml_features.parquet"

INPUT_SCHEMA_AUDIT_PATH = OUTPUT_DIR / "05_input_schema_audit.csv"
FEATURE_GROUPS_CSV_PATH = OUTPUT_DIR / "05_ml_weather_feature_groups.csv"
FEATURE_GROUPS_JSON_PATH = OUTPUT_DIR / "05_ml_weather_feature_groups.json"
WINDOW_SUMMARY_PATH = OUTPUT_DIR / "05_operational_window_summary.csv"
HORIZON_SOURCE_SUMMARY_PATH = OUTPUT_DIR / "05_horizon_source_summary.csv"
MISSINGNESS_CHECK_PATH = OUTPUT_DIR / "05_missingness_check.csv"
LEAKAGE_CHECK_PATH = OUTPUT_DIR / "05_leakage_column_check.csv"
VALIDATION_CHECKS_PATH = OUTPUT_DIR / "05_validation_checks.csv"
END_SAMPLE_ROW_REDUCTION_AUDIT_PATH = OUTPUT_DIR / "05_end_sample_row_reduction_audit.csv"

MAIN_OPERATIONAL_WEATHER_WINDOW = "trade_08_19"
ROBUSTNESS_WEATHER_WINDOW = "trade_08_18"
BENCHMARK_WEATHER_WINDOW = "full_day_00_23"
EXPECTED_WINDOWS = [
    BENCHMARK_WEATHER_WINDOW,
    ROBUSTNESS_WEATHER_WINDOW,
    MAIN_OPERATIONAL_WEATHER_WINDOW,
]
EXPECTED_HORIZONS = list(range(0, 11))
SYNTHETIC_EMULATOR_SOURCE = "synthetic_realised_anchor_error_calibrated"

KEY_COLUMNS = [
    "origin_date",
    "origin_hour",
    "origin_datetime_utc",
    "target_date",
    "horizon_days",
    "AvdelingID",
    "aggregation_window",
]
AUDIT_METADATA_COLUMNS = [
    "forecast_source",
    "forecast_support_level",
    "calibration_source",
    "uncertainty_source",
    "calibration_fallback_level",
    "uncertainty_fallback_level",
    "climatology_fallback_level",
    "h2_anchor_weight",
    "climatology_weight",
    "uncertainty_scale",
]
M2_POINT_WEATHER_FEATURES = [
    "temp_fcst_mean",
    "precip_fcst_sum",
    "wind_fcst_mean",
    "humid_fcst_mean",
    "cloud_fcst_mean",
]
M4_UNCERTAINTY_FEATURES = [
    "temp_fcst_std",
    "temp_fcst_p10",
    "temp_fcst_p90",
    "temp_fcst_interval_width",
    "wind_fcst_std",
    "wind_fcst_p10",
    "wind_fcst_p90",
    "wind_fcst_interval_width",
    "humid_fcst_std",
    "humid_fcst_p10",
    "humid_fcst_p90",
    "humid_fcst_interval_width",
    "cloud_fcst_std",
    "cloud_fcst_p10",
    "cloud_fcst_p90",
    "cloud_fcst_interval_width",
    "precip_wet_prob",
    "precip_amount_uncertainty",
    "precip_fcst_p10",
    "precip_fcst_p90",
    "precip_fcst_interval_width",
    "precip_wet_amount_p90",
    "cloud_open_prob",
    "cloud_partly_prob",
    "cloud_overcast_prob",
]
M4_WEATHER_FEATURES = M2_POINT_WEATHER_FEATURES + M4_UNCERTAINTY_FEATURES
FORBIDDEN_ML_FEATURE_COLUMNS = ["apparent_temp_fcst_mean"]
ML_VIEW_COLUMNS = KEY_COLUMNS + AUDIT_METADATA_COLUMNS + M4_WEATHER_FEATURES
READ_COLUMNS = sorted(set(ML_VIEW_COLUMNS + ["apparent_temp_fcst_mean"]))

print(f"Project root: {PROJECT_ROOT}")
print(f"Main operational weather window: {MAIN_OPERATIONAL_WEATHER_WINDOW}")
print(f"Operational forecast input: {project_relative(OPERATIONAL_FORECAST_PATH)}")
print(f"ML-ready weather feature output: {project_relative(ML_WEATHER_FEATURES_PATH)}")

require_file(OPERATIONAL_FORECAST_PATH, "operational forecast output from notebook 04")
require_file(ERROR_DIAGNOSTICS_PATH, "diagnostics-only forecast-error output from notebook 04")

schema_rows = []
for name, path in [
    ("operational_forecast_windows", OPERATIONAL_FORECAST_PATH),
    ("forecast_error_diagnostics_windows", ERROR_DIAGNOSTICS_PATH),
]:
    meta = pq.ParquetFile(path)
    schema_rows.append(
        {
            "input": name,
            "path": project_relative(path),
            "rows": int(meta.metadata.num_rows),
            "columns": int(len(meta.schema_arrow.names)),
            "size_bytes": int(path.stat().st_size),
            "column_names": ",".join(meta.schema_arrow.names),
        }
    )
input_schema_audit = pd.DataFrame(schema_rows)
input_schema_audit.to_csv(INPUT_SCHEMA_AUDIT_PATH, index=False)
display(input_schema_audit[["input", "rows", "columns", "size_bytes"]])

operational = pd.read_parquet(OPERATIONAL_FORECAST_PATH, columns=READ_COLUMNS)
require_columns(operational, READ_COLUMNS, "operational forecast input")
for date_col in ["origin_date", "origin_datetime_utc", "target_date"]:
    operational[date_col] = pd.to_datetime(operational[date_col])
operational["origin_datetime_utc"] = pd.to_datetime(operational["origin_datetime_utc"], utc=True)
operational["origin_hour"] = operational["origin_hour"].astype("int16")
operational["horizon_days"] = operational["horizon_days"].astype("int16")
operational["AvdelingID"] = operational["AvdelingID"].astype("int64")

available_windows = sorted(operational["aggregation_window"].dropna().astype(str).unique().tolist())
if available_windows != EXPECTED_WINDOWS:
    raise AssertionError(
        f"Unexpected aggregation windows {available_windows}; expected {EXPECTED_WINDOWS}"
    )
available_horizons = sorted(int(x) for x in operational["horizon_days"].dropna().unique())
if available_horizons != EXPECTED_HORIZONS:
    raise AssertionError(f"Unexpected horizons {available_horizons}; expected {EXPECTED_HORIZONS}")

window_summary = (
    operational.groupby("aggregation_window", observed=True)
    .agg(
        rows=("AvdelingID", "size"),
        stores=("AvdelingID", "nunique"),
        origin_date_min=("origin_date", "min"),
        origin_date_max=("origin_date", "max"),
        target_date_min=("target_date", "min"),
        target_date_max=("target_date", "max"),
        horizon_min=("horizon_days", "min"),
        horizon_max=("horizon_days", "max"),
    )
    .reset_index()
)
window_summary.to_csv(WINDOW_SUMMARY_PATH, index=False)

ml_weather_features = operational.loc[
    operational["aggregation_window"].astype(str).eq(MAIN_OPERATIONAL_WEATHER_WINDOW),
    ML_VIEW_COLUMNS,
].copy()
ml_weather_features["aggregation_window"] = ml_weather_features["aggregation_window"].astype("string")

key_without_window = [
    "origin_date",
    "origin_hour",
    "origin_datetime_utc",
    "target_date",
    "horizon_days",
    "AvdelingID",
]
duplicate_count = int(ml_weather_features.duplicated(key_without_window).sum())
if duplicate_count:
    raise AssertionError(f"Duplicate ML weather feature keys after main-window selection: {duplicate_count}")

selected_horizons = sorted(int(x) for x in ml_weather_features["horizon_days"].dropna().unique())
if selected_horizons != EXPECTED_HORIZONS:
    raise AssertionError(
        f"Main ML view has horizons {selected_horizons}; expected {EXPECTED_HORIZONS}"
    )

h012_sources = sorted(
    ml_weather_features.loc[
        ml_weather_features["horizon_days"].isin([0, 1, 2]),
        "forecast_source",
    ]
    .unique()
    .tolist()
)
h310_sources = sorted(
    ml_weather_features.loc[
        ml_weather_features["horizon_days"].between(3, 10),
        "forecast_source",
    ]
    .unique()
    .tolist()
)
if h012_sources != ["deterministic_meps"]:
    raise AssertionError(f"h=0/1/2 should be deterministic_meps, found {h012_sources}")
if h310_sources != [SYNTHETIC_EMULATOR_SOURCE]:
    raise AssertionError(f"h=3..10 should be synthetic extension, found {h310_sources}")

leakage_columns = [
    col for col in ml_weather_features.columns
    if col.endswith("_obs") or "_obs_" in col or col.endswith("_error") or "forecast_error" in col
]
pressure_columns = [
    col for col in ml_weather_features.columns
    if "pressure" in col.lower() or "mslp" in col.lower()
]
apparent_feature_violations = [
    col
    for col in FORBIDDEN_ML_FEATURE_COLUMNS
    if col in M2_POINT_WEATHER_FEATURES or col in M4_WEATHER_FEATURES
]
if leakage_columns:
    raise AssertionError(f"ML-ready weather feature view contains leakage/diagnostic columns: {leakage_columns}")
if pressure_columns:
    raise AssertionError(
        "Pressure/MSLP columns must not be retained in the ML-ready weather feature view: "
        f"{pressure_columns}"
    )
if apparent_feature_violations:
    raise AssertionError(f"Apparent temperature is incorrectly listed as an ML feature: {apparent_feature_violations}")
if any(col in ml_weather_features.columns for col in FORBIDDEN_ML_FEATURE_COLUMNS):
    raise AssertionError("apparent_temp_fcst_mean should not be retained in the ML-ready weather feature view")

m2_missing = ml_weather_features[M2_POINT_WEATHER_FEATURES].isna().sum()
m4_missing = ml_weather_features[M4_WEATHER_FEATURES].isna().sum()
if int(m2_missing.sum()) != 0:
    raise AssertionError(f"M2 weather features contain missing values: {m2_missing[m2_missing.gt(0)].to_dict()}")
if int(m4_missing.sum()) != 0:
    raise AssertionError(f"M4 weather features contain missing values: {m4_missing[m4_missing.gt(0)].to_dict()}")

feature_group_rows = []
for col in M2_POINT_WEATHER_FEATURES:
    feature_group_rows.append(
        {
            "column": col,
            "role": "feature",
            "feature_group": "m2_point_weather",
            "allowed_m2_operational_point_weather": True,
            "allowed_m4_operational_uncertainty_weather": True,
            "known_at_origin": True,
            "leakage_risk": False,
            "retained_in_ml_weather_view": True,
            "notes": "Operational point forecast weather feature from notebook 04.",
        }
    )
for col in M4_UNCERTAINTY_FEATURES:
    feature_group_rows.append(
        {
            "column": col,
            "role": "feature",
            "feature_group": "m4_uncertainty_probability_interval_weather",
            "allowed_m2_operational_point_weather": False,
            "allowed_m4_operational_uncertainty_weather": True,
            "known_at_origin": True,
            "leakage_risk": False,
            "retained_in_ml_weather_view": True,
            "notes": "Operational uncertainty/probability/interval feature from notebook 04.",
        }
    )
for col in KEY_COLUMNS:
    feature_group_rows.append(
        {
            "column": col,
            "role": "key",
            "feature_group": "forecast_key",
            "allowed_m2_operational_point_weather": False,
            "allowed_m4_operational_uncertainty_weather": False,
            "known_at_origin": col not in ["target_date"],
            "leakage_risk": False,
            "retained_in_ml_weather_view": True,
            "notes": "Join key or explicit weather-window identifier, not a default ML feature.",
        }
    )
for col in AUDIT_METADATA_COLUMNS:
    feature_group_rows.append(
        {
            "column": col,
            "role": "metadata",
            "feature_group": "forecast_audit_metadata",
            "allowed_m2_operational_point_weather": False,
            "allowed_m4_operational_uncertainty_weather": False,
            "known_at_origin": True,
            "leakage_risk": False,
            "retained_in_ml_weather_view": True,
            "notes": "Retained for audit/stratification only; not a default ML feature.",
        }
    )
feature_groups = pd.DataFrame(feature_group_rows)
feature_groups.to_csv(FEATURE_GROUPS_CSV_PATH, index=False)
FEATURE_GROUPS_JSON_PATH.write_text(
    json.dumps(
        {
            "main_operational_weather_window": MAIN_OPERATIONAL_WEATHER_WINDOW,
            "m2_point_weather_features": M2_POINT_WEATHER_FEATURES,
            "m4_weather_features": M4_WEATHER_FEATURES,
            "audit_metadata_columns": AUDIT_METADATA_COLUMNS,
            "forbidden_default_ml_features": FORBIDDEN_ML_FEATURE_COLUMNS,
        },
        indent=2,
    ),
    encoding="utf-8",
)

horizon_source_summary = (
    ml_weather_features.groupby(["horizon_days", "forecast_source", "forecast_support_level"], observed=True)
    .agg(
        rows=("AvdelingID", "size"),
        stores=("AvdelingID", "nunique"),
        origin_date_min=("origin_date", "min"),
        origin_date_max=("origin_date", "max"),
        target_date_min=("target_date", "min"),
        target_date_max=("target_date", "max"),
    )
    .reset_index()
)
horizon_source_summary.to_csv(HORIZON_SOURCE_SUMMARY_PATH, index=False)

reference_rows_h0 = int(ml_weather_features.loc[ml_weather_features["horizon_days"].eq(0)].shape[0])
end_sample_row_reduction_audit = horizon_source_summary.copy()
end_sample_row_reduction_audit["reference_rows_h0"] = reference_rows_h0
end_sample_row_reduction_audit["inferred_missing_rows_vs_h0"] = (
    (reference_rows_h0 - end_sample_row_reduction_audit["rows"])
    .clip(lower=0)
    .astype("int64")
)
end_sample_row_reduction_audit["inferred_reason"] = np.where(
    (end_sample_row_reduction_audit["horizon_days"].astype(int) >= 3)
    & end_sample_row_reduction_audit["inferred_missing_rows_vs_h0"].gt(0),
    "missing realised target-day latent weather in notebook 04 near sample end",
    "none inferred",
)
end_sample_row_reduction_audit.to_csv(END_SAMPLE_ROW_REDUCTION_AUDIT_PATH, index=False)

missingness_rows = []
for col in M2_POINT_WEATHER_FEATURES + M4_UNCERTAINTY_FEATURES + AUDIT_METADATA_COLUMNS:
    missing = int(ml_weather_features[col].isna().sum())
    missingness_rows.append(
        {
            "column": col,
            "missing_values": missing,
            "missing_share": float(missing / len(ml_weather_features)) if len(ml_weather_features) else np.nan,
        }
    )
missingness_check = pd.DataFrame(missingness_rows)
missingness_check.to_csv(MISSINGNESS_CHECK_PATH, index=False)

leakage_check = pd.DataFrame(
    [
        {
            "check": "leakage_like_columns",
            "value": ",".join(leakage_columns),
            "violation_count": len(leakage_columns),
        },
        {
            "check": "apparent_temperature_in_m2_feature_list",
            "value": str("apparent_temp_fcst_mean" in M2_POINT_WEATHER_FEATURES),
            "violation_count": int("apparent_temp_fcst_mean" in M2_POINT_WEATHER_FEATURES),
        },
        {
            "check": "apparent_temperature_in_m4_feature_list",
            "value": str("apparent_temp_fcst_mean" in M4_WEATHER_FEATURES),
            "violation_count": int("apparent_temp_fcst_mean" in M4_WEATHER_FEATURES),
        },
        {
            "check": "apparent_temperature_in_ml_weather_view",
            "value": str("apparent_temp_fcst_mean" in ml_weather_features.columns),
            "violation_count": int("apparent_temp_fcst_mean" in ml_weather_features.columns),
        },
        {"check": "forecast_error_diagnostics_merged", "value": "False", "violation_count": 0},
        {
            "check": "pressure_or_mslp_columns",
            "value": ",".join(pressure_columns),
            "violation_count": len(pressure_columns),
        },
    ]
)
leakage_check.to_csv(LEAKAGE_CHECK_PATH, index=False)

validation_checks = pd.DataFrame([
    {"check": "input_path", "value": project_relative(OPERATIONAL_FORECAST_PATH)},
    {"check": "main_window", "value": MAIN_OPERATIONAL_WEATHER_WINDOW},
    {"check": "rows", "value": int(len(ml_weather_features))},
    {"check": "columns", "value": int(ml_weather_features.shape[1])},
    {"check": "duplicate_key_count", "value": duplicate_count},
    {"check": "horizons", "value": ",".join(str(h) for h in selected_horizons)},
    {"check": "h0_h1_h2_source", "value": ",".join(h012_sources)},
    {"check": "h3_h10_source", "value": ",".join(h310_sources)},
    {"check": "m2_missing_values", "value": int(m2_missing.sum())},
    {"check": "m4_missing_values", "value": int(m4_missing.sum())},
    {"check": "leakage_like_columns", "value": ",".join(leakage_columns)},
    {"check": "pressure_or_mslp_columns", "value": ",".join(pressure_columns)},
    {"check": "apparent_temperature_excluded_from_feature_lists", "value": True},
])
validation_checks.to_csv(VALIDATION_CHECKS_PATH, index=False)

ml_weather_features.to_parquet(ML_WEATHER_FEATURES_PATH, index=False)

print(
    f"Wrote {project_relative(ML_WEATHER_FEATURES_PATH)} "
    f"with {len(ml_weather_features):,} rows and {ml_weather_features.shape[1]} columns"
)
print(f"Wrote feature-group metadata: {project_relative(FEATURE_GROUPS_CSV_PATH)}")
print("Validation checks:")
display(validation_checks)
print("Feature groups:")
display(feature_groups.groupby(["role", "feature_group"], observed=True).size().rename("columns").reset_index())


## Downstream contract

Notebook 06 should use `data/processed/weather_forecast_operational_ml_features.parquet` as the operational weather feature input, joined by forecast origin, target date, horizon, and store. The feature-group metadata separates M2 point-weather features from M4 uncertainty/probability/interval features and keeps audit metadata out of default ML feature sets.
